Instructions on how to submit predictions to a Kaggle Competition via their API. More information [here](https://github.com/Kaggle/kaggle-api).

I will go into the details of the modelling process in a future post. This post is to simply highlight submission process.

## 1. Get Data 

In [ ]:
from pathlib import Path
import zipfile,kaggle
path = Path('titanic')
kaggle.api.competition_download_cli(str(path))
zipfile.ZipFile(f'{path}.zip').extractall(path)

In [ ]:
from fastai.tabular.all import *
pd.options.display.float_format = '{:.2f}'.format
set_seed(12)

In [ ]:
df = pd.read_csv(path/'train.csv')

## 2. Prepare Data

### 2.1 Feature Engineering (FE)

I'll go into the details in a future post.

In [ ]:
def add_features(df):
    df['LogFare'] = np.log1p(df['Fare'])
    df['Deck'] = df.Cabin.str[0].map(dict(A="ABC", B="ABC", C="ABC", D="DE", E="DE", F="FG", G="FG"))
    df['Family'] = df.SibSp+df.Parch
    df['Alone'] = df.Family==1
    df['TicketFreq'] = df.groupby('Ticket')['Ticket'].transform('count')
    df['Title'] = df.Name.str.split(', ', expand=True)[1].str.split('.', expand=True)[0]
    df['Title'] = df.Title.map(dict(Mr="Mr",Miss="Miss",Mrs="Mrs",Master="Master")).value_counts(dropna=False)

add_features(df)

### 2.2 Test and Validation sets

In [ ]:
splits = RandomSplitter(seed=12)(df)

### 2.3 Create Tabular DataLoader

In [ ]:
dls = TabularPandas(
    df, splits=splits,
    procs = [Categorify, FillMissing, Normalize],
    cat_names=["Sex","Pclass","Embarked","Deck", "Title"],
    cont_names=['Age', 'SibSp', 'Parch', 'LogFare', 'Alone', 'TicketFreq', 'Family'],
    y_names="Survived", y_block = CategoryBlock(),
).dataloaders(path=".")

## 3. Train our Learner

### 3.1 Create Learner

In [ ]:
titanic_learner = tabular_learner(dls, metrics=accuracy, layers=[10,10])

### 3.2 Learner Rate (LR)

In [ ]:
titanic_learner.lr_find(suggest_funcs=(slide, valley))

### 3.3 Training

In [ ]:
titanic_learner.fit(16, lr=0.03)

## 4. Predictions

### 4.1 Import Test Data and Apply FE

In [ ]:
tst_df = pd.read_csv(path/'test.csv')
# tst_df['Fare'] = tst_df.Fare.fillna(0)
# add_features(tst_df)